In [ ]:
!pip install torch torchvision pillow scikit-learn tqdm matplotlib -q
import os, urllib.request, zipfile, shutil

url = "http://www.cedar.buffalo.edu/NIJ/data/signatures.rar"
!wget -q http://www.cedar.buffalo.edu/NIJ/data/signatures.rar -O signatures.rar
!apt-get install -y unrar -q && unrar x signatures.rar

for root, dirs, files in os.walk("."):
    for d in dirs:
        if "org" in d.lower() or "forg" in d.lower():
            print(os.path.join(root, d), "→", len(os.listdir(os.path.join(root, d))), "files")

import os, random
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

DATA_DIR      = "/content/signatures"   # adjust if your dataset lives elsewhere
GENUINE_DIR   = os.path.join(DATA_DIR, "full_org")
FORGE_DIR     = os.path.join(DATA_DIR, "full_forg")
SAVE_PATH     = "/content/siamese_best.pth"
IMG_SIZE      = 105
EMBED_DIM     = 128
BATCH_SIZE    = 32
EPOCHS        = 20
LR            = 1e-4
MARGIN        = 1.0
MAX_PAIRS     = 100
THRESHOLD     = 0.5
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

def build_pairs(genuine_dir, forge_dir, max_pairs=100, seed=42):
    random.seed(seed)
    pairs, labels = [], []
    signers = {}
    for fname in sorted(os.listdir(genuine_dir)):
        if not fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
            continue
        parts = fname.rsplit(".", 1)[0].split("_")
        sid = parts[1] if len(parts) >= 3 else parts[0]
        signers.setdefault(sid, []).append(os.path.join(genuine_dir, fname))

    for sid, paths in signers.items():
        pos = list(combinations(paths, 2))[:max_pairs]
        pairs += pos;  labels += [1] * len(pos)

        forg_paths = [os.path.join(forge_dir, f)
                      for f in os.listdir(forge_dir)
                      if len(f.split("_")) > 1 and f.split("_")[1] == sid and
                         f.lower().endswith((".png",".jpg",".jpeg",".bmp"))]
        if not forg_paths:
            continue
        neg = [(g, f) for g in paths for f in forg_paths]
        neg = random.sample(neg, min(max_pairs, len(neg)))
        pairs += neg;  labels += [0] * len(neg)

    print(f"Pairs — total:{len(pairs)}  genuine:{sum(labels)}  forged:{len(labels)-sum(labels)}")
    return pairs, labels

# DATASET
def get_transform(augment=False):
    ops = [T.Grayscale(3), T.Resize((IMG_SIZE, IMG_SIZE))]
    if augment:
        ops += [T.RandomHorizontalFlip(), T.RandomRotation(10)]
    ops += [T.ToTensor(),
            T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])]
    return T.Compose(ops)

class SiameseDataset(Dataset):
    def __init__(self, pairs, labels, augment=False):
        self.pairs = pairs;  self.labels = labels
        self.tf = get_transform(augment)
    def __len__(self):  return len(self.pairs)
    def __getitem__(self, i):
        a = self.tf(Image.open(self.pairs[i][0]).convert("RGB"))
        b = self.tf(Image.open(self.pairs[i][1]).convert("RGB"))
        return a, b, torch.tensor(self.labels[i], dtype=torch.float32)

# MODEL
class SiameseNet(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        bb = models.resnet18(weights="IMAGENET1K_V1")
        for n, p in bb.named_parameters():
            if "layer1" in n or "layer2" in n:
                p.requires_grad = False
        bb.fc = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, embed_dim))
        self.bb = bb

    def forward_one(self, x):  return self.bb(x)
    def forward(self, x1, x2):
        return self.forward_one(x1), self.forward_one(x2)

# ─── 4.
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__();  self.margin = margin
    def forward(self, e1, e2, y):
        d = F.pairwise_distance(e1, e2)
        loss = y*d**2 + (1-y)*torch.clamp(self.margin - d, min=0)**2
        return loss.mean()

# ─── 5.
pairs, labels = build_pairs(GENUINE_DIR, FORGE_DIR, MAX_PAIRS)
ds = SiameseDataset(pairs, labels, augment=True)
n_train = int(0.8 * len(ds))
train_ds, val_ds = random_split(ds, [n_train, len(ds) - n_train])
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model     = SiameseNet(EMBED_DIM).to(DEVICE)
criterion = ContrastiveLoss(MARGIN)
opt       = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
sched     = torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)

train_losses, val_losses = [], []
best_val = float("inf")

for epoch in range(1, EPOCHS+1):
    model.train();  tl = 0
    for a, b, y in tqdm(train_dl, desc=f"Epoch {epoch:02d}", leave=False):
        a, b, y = a.to(DEVICE), b.to(DEVICE), y.to(DEVICE)
        e1, e2 = model(a, b)
        loss = criterion(e1, e2, y)
        opt.zero_grad();  loss.backward();  opt.step()
        tl += loss.item()
    tl /= len(train_dl)

    model.eval();  vl = 0
    with torch.no_grad():
        for a, b, y in val_dl:
            e1, e2 = model(a.to(DEVICE), b.to(DEVICE))
            vl += criterion(e1, e2, y.to(DEVICE)).item()
    vl /= len(val_dl)
    sched.step()

    train_losses.append(tl);  val_losses.append(vl)
    print(f"Epoch {epoch:02d} | train={tl:.4f}  val={vl:.4f}")
    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  ✓ saved (val={vl:.4f})")

# ─── 6.
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train loss")
plt.plot(val_losses,   label="Val loss")
plt.xlabel("Epoch");  plt.ylabel("Contrastive loss")
plt.title("Siamese network training");  plt.legend();  plt.tight_layout()
plt.savefig("/content/loss_curve.png", dpi=150)
plt.show()

# ─── 7.
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

all_dists, all_labels = [], []
with torch.no_grad():
    for a, b, y in val_dl:
        e1, e2 = model(a.to(DEVICE), b.to(DEVICE))
        all_dists.extend(F.pairwise_distance(e1, e2).cpu().numpy())
        all_labels.extend(y.numpy())

dists  = np.array(all_dists)
labels_arr = np.array(all_labels)
preds  = (dists < THRESHOLD).astype(int)

acc = accuracy_score(labels_arr, preds)
tn, fp, fn, tp = confusion_matrix(labels_arr, preds).ravel()
FAR = fp / (fp + tn) if (fp+tn) > 0 else 0
FRR = fn / (fn + tp) if (fn+tp) > 0 else 0
auc = roc_auc_score(labels_arr, -dists)

print("\n" + "═"*40)
print(f"  Accuracy : {acc*100:.2f}%")
print(f"  FAR      : {FAR*100:.2f}%")
print(f"  FRR      : {FRR*100:.2f}%")
print(f"  AUC-ROC  : {auc:.4f}")
print("═"*40)

# ─── 8.
def predict(img1_path, img2_path, threshold=THRESHOLD):
    tf = get_transform(augment=False)
    model.eval()
    with torch.no_grad():
        t1 = tf(Image.open(img1_path).convert("RGB")).unsqueeze(0).to(DEVICE)
        t2 = tf(Image.open(img2_path).convert("RGB")).unsqueeze(0).to(DEVICE)
        e1, e2 = model(t1, t2)
        dist = F.pairwise_distance(e1, e2).item()
    verdict = "GENUINE ✅" if dist < threshold else "FORGED ❌"
    print(f"Distance: {dist:.4f}  |  Threshold: {threshold}  |  → {verdict}")

    fig, ax = plt.subplots(1, 2, figsize=(6, 3))
    ax[0].imshow(Image.open(img1_path), cmap="gray");  ax[0].set_title("Reference");  ax[0].axis("off")
    ax[1].imshow(Image.open(img2_path), cmap="gray");  ax[1].set_title(f"{verdict}");  ax[1].axis("off")
    plt.tight_layout();  plt.show()
    return dist, verdict

# Demo
sample_genuine = os.listdir(GENUINE_DIR)[0]
sample_forged  = os.listdir(FORGE_DIR)[0]
predict(os.path.join(GENUINE_DIR, sample_genuine),
        os.path.join(FORGE_DIR,   sample_forged))

Reading package lists...
Building dependency tree...
Reading state information...
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from signatures.rar


Would you like to replace the existing file signatures/full_forg/forgeries_10_1.png
 53878 bytes, modified on 2004-01-28 12:02
with a new one
 53878 bytes, modified on 2004-01-28 12:02

[Y]es, [N]o, [A]ll, n[E]ver, [R]ename, [Q]uit nEver


Would you like to replace the existing file signatures/full_forg/forgeries_10_10.png
 43281 bytes, modified on 2004-01-28 12:01
with a new one
 43281 bytes, modified on 2004-01-28 12:01

[Y]es, [N]o, [A]ll, n[E]ver, [R]ename, [Q]uit Quit

Program aborted
./signatures/full_forg → 1321 files
./signatures/full_org → 1321 files
Device: cuda
Pairs — total:11000  genuine:5500  forged:5500


Epoch 01:  16%|█▋        | 45/275 [00:14<01:49,  2.11it/s]